# sparse_knn — generation and scoring

---
## 1 — Host and working tree

In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

name, memory.total [MiB], memory.used [MiB], driver_version
NVIDIA GeForce RTX 4090, 24564 MiB, 1 MiB, 580.95.05


In [5]:
import os

if not os.path.isdir('Style-Aware-MT') and not os.path.exists('manage.py'):
    !git clone https://github.com/prnamhr/Style-Aware-MT.git
if os.path.isdir('Style-Aware-MT'):
    %cd Style-Aware-MT
!git rev-parse --short HEAD

2fb5200


In [3]:
%pip install -r requirements.txt

  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 1.4 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 1.8 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 62.0 MB/s  0:00:19m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 65.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 53.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 72.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 54.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 64.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 61.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.4/

In [4]:
# The pipeline is text-only and these three ship against a torch the pins contradict.
%pip uninstall -q -y torchvision torchaudio torchcodec

Note: you may need to restart the kernel to use updated packages.


In [6]:
import torch

cap = torch.cuda.get_device_capability(0)
print(f'{torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}',
      f'torch {torch.__version__} / cuda {torch.version.cuda}')
assert torch.cuda.is_bf16_supported(), 'bf16 unsupported; the frozen base is not quantized'

NVIDIA GeForce RTX 4090  sm_89 torch 2.12.0+cu130 / cuda 13.0


---
## 2 — Run parameters and the matched-contrast gate

In [7]:
import hashlib
import json
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import yaml

PY = sys.executable
COMET_PY = '.venv-comet/bin/python'
SPLIT = 'val'
ARM, BASELINE = 'sparse_knn', 'knn_fewshot'
CONFIG, BASE_CONFIG = Path('configs/sparse_knn.yaml'), Path('configs/base_qwen.yaml')
OUT = Path('outputs')

DIAG = Path(f'results/sparse_selection_{SPLIT}.json')
ROUTING = Path(f'results/sparse_routing_{SPLIT}.json')

N_BOOT, N_STYLO, ALPHA, SEED = 10000, 2000, 0.05, 42

CFG = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
RETR, SPA, RAR, PROMPT = CFG['retrieval'], CFG['sparse'], CFG['rarity'], CFG['prompt']
RARITY_PATH = Path(RAR['out'])
print(f"{ARM} against {BASELINE}: k={RETR['k']}, m={SPA['m']}, "
      f"df band [{RAR['df_min']}, {RAR['df_max']}], min_query_terms={SPA['min_query_terms']}")

sparse_knn against knn_fewshot: k=8, m=4, df band [2, 20], min_query_terms=1


In [8]:
BASE = yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))
for block in ('generator', 'prompt', 'retrieval', 'data', 'output'):
    assert CFG[block] == BASE[block], f'{block} differs from {BASE_CONFIG}: {CFG[block]}'
assert set(CFG) - set(BASE) == {'rarity', 'sparse'}, sorted(set(CFG) - set(BASE))
assert set(BASE) - set(CFG) == {'afsp'}, sorted(set(BASE) - set(CFG))
print(f'{CONFIG.name} differs from {BASE_CONFIG.name} in the selection blocks only')

sparse_knn.yaml differs from base_qwen.yaml in the selection blocks only


In [9]:
VAL = [json.loads(x) for x in Path(CFG['data']['eval_file']).open(encoding='utf-8') if x.strip()]
SRC = [r['input'] for r in VAL]

BASE_ROWS = [json.loads(x) for x in (OUT / f'{BASELINE}_{SPLIT}.jsonl').open(encoding='utf-8')
             if x.strip()]
assert len(BASE_ROWS) == len(VAL), f'{BASELINE}: {len(BASE_ROWS)} rows, expected {len(VAL)}'
assert [r['input'] for r in BASE_ROWS] == SRC, f'{BASELINE} is not aligned to {SPLIT}.jsonl'
assert all(r['model'] == CFG['generator']['model'] for r in BASE_ROWS), 'a different base model'
print(f'{len(VAL)} {SPLIT} segments; {BASELINE} present and aligned on {BASE_ROWS[0]["model"]}')

1323 val segments; knn_fewshot present and aligned on Qwen/Qwen2.5-7B-Instruct


---
## 3 — The rarity list and the index

In [10]:
RARITY = json.loads(RARITY_PATH.read_text(encoding='utf-8'))
RARITY_SHA = hashlib.sha256(RARITY_PATH.read_bytes()).hexdigest()

assert RARITY['config']['df_min'] == RAR['df_min'], RARITY['config']
assert RARITY['config']['df_max'] == RAR['df_max'], RARITY['config']
assert RARITY['df_histogram']['irregular']['1'] == 0, 'hapaxes are back in the list'
assert RARITY['df_observed'] == [RAR['df_min'], RAR['df_max']], RARITY['df_observed']
print(f"{RARITY['n_irregular']} irregular terms, df {RARITY['df_observed']}, "
      f"{RARITY['selected_frac']:.1%} of the vocabulary")
print(f'sha256 {RARITY_SHA[:16]}...')

10061 irregular terms, df [2, 20], 44.1% of the vocabulary
sha256 47da96ea667ee095...


In [12]:
# data/knn_index is git-ignored, so a fresh session rebuilds it. It must be the same
# index knn_fewshot retrieved from, hence base_qwen.yaml rather than the arm's config.
INDEX = Path(RETR['index_dir'])
INDEX_FILES = ('embeddings.npy', 'pairs.jsonl', 'meta.json')
if not all((INDEX / f).exists() for f in INDEX_FILES):
    !python3 manage.py build_index --config configs/base_qwen.yaml

meta = json.loads((INDEX / 'meta.json').read_text(encoding='utf-8'))
assert meta['embed_model'] == RETR['embed_model'], meta
INDEX_SHA = {f: hashlib.sha256((INDEX / f).read_bytes()).hexdigest()[:16] for f in INDEX_FILES}
print(f"{meta['n_passages']} pool passages on {meta['embed_model']}, dim {meta['dim']}")
print(json.dumps(INDEX_SHA, indent=2))

10860 pool passages on intfloat/multilingual-e5-large-instruct, dim 1024
{
  "embeddings.npy": "9c282c8042ab740e",
  "pairs.jsonl": "c48f429809435593",
  "meta.json": "b028a2a81f9f0eed"
}


---
## 4 — Routing, recorded per segment


In [13]:
from src.retrieval.rarity import load_irregular
from src.retrieval.retrieve import RetrievalIndex
from src.retrieval.sparse import SparseRetriever

index = RetrievalIndex(RETR['index_dir'], embed_model=RETR['embed_model'])
retriever = SparseRetriever(
    index, load_irregular(str(RARITY_PATH)), index,
    zwnj=RAR['zwnj'], m=SPA['m'], redundancy=SPA['redundancy'],
    min_query_terms=SPA['min_query_terms'],
)
SELECTED, TRACES = retriever.select_with_trace(SRC, k=RETR['k'])
print(f'{len(TRACES)} traces, {len(SELECTED[0])} exemplars per prompt')

Loading intfloat/multilingual-e5-large-instruct on device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/140k [00:00<?, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

1323 traces, 8 exemplars per prompt


In [14]:
SLOTS = min(SPA['m'], RETR['k'])
HIST = {str(v): sum(t['n_sparse'] == v for t in TRACES) for v in range(SLOTS + 1)}
ROUTES = {r: sum(t['route'] == r for t in TRACES) for r in ('full', 'partial', 'dense')}

diag = json.loads(DIAG.read_text(encoding='utf-8'))
assert diag['config']['index_dir'] == RETR['index_dir'], diag['config']
assert HIST == diag['n_sparse']['histogram'], f'{HIST} against {diag["n_sparse"]["histogram"]}'
assert ROUTES == diag['routes'], f'{ROUTES} against {diag["routes"]}'

ROUTING.write_text(json.dumps({
    'split': SPLIT,
    'index_dir': RETR['index_dir'],
    'rarity_sha256': RARITY_SHA,
    'k': RETR['k'],
    'm': SPA['m'],
    'routes': ROUTES,
    'n_sparse_histogram': HIST,
    'segments': [{'route': t['route'], 'n_sparse': t['n_sparse'], 'coverage': t['coverage'],
                  'n_query_terms': t['n_query_terms']} for t in TRACES],
}, ensure_ascii=False, indent=2), encoding='utf-8')

routed = len(TRACES) - ROUTES['dense']
print(f'routes {ROUTES}  ({routed / len(TRACES):.1%} routed)')
print(f'rarity slots filled {HIST}, mean {np.mean([t["n_sparse"] for t in TRACES]):.3f}')
print(f'matches {DIAG}; wrote {ROUTING}')

routes {'full': 483, 'partial': 724, 'dense': 116}  (91.2% routed)
rarity slots filled {'0': 116, '1': 261, '2': 250, '3': 213, '4': 483}, mean 2.519
matches results/sparse_selection_val.json; wrote results/sparse_routing_val.json


---
## 5 — Stage B: the assembled prompt

In [15]:
from src.infer.run import _load_configured_glossary, build_fewshot_user, order_exemplars

K = RETR['k']
STYLE = Path(PROMPT['style_instruction_file']).read_text(encoding='utf-8')
GLOSSARY = _load_configured_glossary(CFG)
ORDERED = [order_exemplars(ex, PROMPT['ordering']) for ex in SELECTED]
PROMPTS = [build_fewshot_user(s, ex, GLOSSARY) for s, ex in zip(SRC, ORDERED)]


def terms_of(text):
    """The irregular terms a string carries, as the channel itself tokenises it."""
    return {retriever.terms[c] for c in retriever.query_terms(text)}


for i, (ex, t) in enumerate(zip(ORDERED, TRACES)):
    keys = [(e['input'], e['output']) for e in ex]
    assert len(keys) == K, f'segment {i}: {len(keys)} exemplars, expected {K}'
    assert len(set(keys)) == len(keys), f'segment {i}: an exemplar repeats across the channels'
    assert SRC[i] not in {e['input'] for e in ex}, f'segment {i}: the query is its own exemplar'
    assert [e['input'] for e in ex] == [index.pairs[r]['input'] for r in t['final_rows'][::-1]], (
        f'segment {i}: prompt order is not the reversed cosine ranking')
    for r in t['sparse_rows']:
        assert terms_of(index.pairs[r]['input']) & set(t['query_terms']), (
            f'segment {i}: rarity pick {r} shares no irregular term with the query')

print(f'{len(PROMPTS)} prompts: {K} distinct exemplars each, cosine-ranked order, '
      f'every rarity pick sharing an irregular term with its query')
print(f'median prompt {int(np.median([len(p) for p in PROMPTS]))} chars, '
      f'longest {max(len(p) for p in PROMPTS)}')

1323 prompts: 8 distinct exemplars each, cosine-ranked order, every rarity pick sharing an irregular term with its query
median prompt 4625 chars, longest 9513


In [16]:
RAR_LEN, COS_LEN = [], []
for t in TRACES:
    rows = set(t['sparse_rows'])
    for r in t['final_rows']:
        (RAR_LEN if r in rows else COS_LEN).append(len(index.pairs[r]['input']))

BASE_SEL = index.retrieve(SRC, k=K)
BASE_CHARS = sum(len(e['input']) for row in BASE_SEL for e in row) / len(TRACES)
ARM_CHARS = (sum(RAR_LEN) + sum(COS_LEN)) / len(TRACES)

for name, v in (('query', [len(x) for x in SRC]), ('rarity channel', RAR_LEN),
                ('cosine fill', COS_LEN)):
    a = np.array(v)
    print(f'{name:16s} n={len(a):6d}  mean {a.mean():6.1f}  median {np.median(a):6.1f} chars')
print(f'exemplar chars per prompt: {BASELINE} {BASE_CHARS:.0f} -> {ARM} {ARM_CHARS:.0f} '
      f'({ARM_CHARS / BASE_CHARS - 1:+.1%})')

query            n=  1323  mean   78.2  median   63.0 chars
rarity channel   n=  3332  mean  109.9  median   83.0 chars
cosine fill      n=  7252  mean  197.0  median  179.0 chars
exemplar chars per prompt: knn_fewshot 1603 -> sparse_knn 1357 (-15.4%)


In [17]:
FILLED = np.array([t['n_sparse'] for t in TRACES])
SHOW = ([int(i) for i in np.flatnonzero(FILLED == SLOTS)[:2]]
        + [int(i) for i in np.flatnonzero((FILLED > 0) & (FILLED < SLOTS))[:2]]
        + [int(i) for i in np.flatnonzero(FILLED == 0)[:1]])
QEMB = index.encode([SRC[i] for i in SHOW])

for q, i in zip(QEMB, SHOW):
    t = TRACES[i]
    rarity = set(t['sparse_rows'])
    print(f"[{i}] {t['route']}  n_sparse {t['n_sparse']}  coverage {t['coverage']}")
    print('  query:', SRC[i][:100])
    print('  terms:', t['query_terms'][:10])
    for pos, row in enumerate(t['final_rows'][::-1], start=1):
        e = index.pairs[row]
        hit = sorted(terms_of(e['input']) & set(t['query_terms']))
        shared = ('+' + ' '.join(hit[:3])) if hit else ''
        print(f"    {pos}. {'rarity' if row in rarity else 'cosine'}  "
              f"cos {float(index.embeddings[row] @ q):.3f}  {shared:24s} {e['input'][:55]}")
    print()

[0] full  n_sparse 4  coverage 0.5912
  query: جواهر الأسرار فی معارج الأسفار لمن اراد ان یتقرّب بالله المقتدر الغفّار فهنیاً للأبرار الّذین یشربون
  terms: ['الأسرار', 'الأسفار', 'الأنهار', 'الغفار', 'معارج', 'یتقرب', 'یشربون']
    1. rarity  cos 0.759  +یتقرب                   کیف یتقرّب الیک و یصعد الی نفسک.
    2. rarity  cos 0.843  +الأسفار                 بشأن الآیة المبارکة فی الأسفار إذا نزلتم واسترحتم المقا
    3. rarity  cos 0.844  +الأسرار                 اخذوا العلوم من الأنبیآء لأنّهم کانوا مطالع الحکمة الال
    4. rarity  cos 0.869  +الأنهار                 و اسألک یا الهی باسمک الّذی به امطرت السّحاب و جرت الأن
    5. cosine  cos 0.879                           من شرب من الکأس الّتی تدور بها ید رحمتک ینقطع عن دونک و
    6. cosine  cos 0.881                           اللّهمّ انّی اسألک بالحرف الّتی اذا خرجت من فم مشیّتک م
    7. cosine  cos 0.882                           فأمطر من سحاب فیض فضلک ما تطهّر به افئدة عبادک عمّا یحج
    8. cosine  cos 0.886                     

In [18]:
i = SHOW[0]
print(STYLE)
print('=' * 88)
print(PROMPTS[i])

You are an expert translator of Bahá'í scripture from Persian and Arabic into English.

Render the source text into English in the formal, elevated, scriptural register of Shoghi Effendi's authorized translations. Observe the following:

- Preserve the dignity and cadence of sacred prose; favour the elevated, archaic register over modern neutral English.
- Use the second-person sacred pronouns and their verb forms where the source addresses the Divine or is addressed by It: "Thou", "Thee", "Thy", "Thine", and verb endings such as "art", "hast", "dost", "doth".
- Retain formal vocatives such as "O" and honorific constructions where the source warrants them.
- Translate the full meaning faithfully; do not add commentary, explanation, transliteration, or footnotes.
- Output only the English translation, as a single continuous passage with no quotation marks, labels, or preamble.

Here are example translations in the required style:

Source: کیف یتقرّب الیک و یصعد الی نفسک.
English: How ca

---
## 6 — Generation

In [20]:
import getpass
import logging
import os

# HF_TOKEN only. The adapter repo is private and the base model is public; nothing in this
# notebook can spend, so a rater key present here would be a mistake, not a convenience.
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY'):
    assert not os.environ.get(var), f'{var} is set; this session makes no paid call'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
logging.getLogger('httpx').setLevel(logging.WARNING)
print('HF_TOKEN set, no rater keys present')

HF_TOKEN:  ········


HF_TOKEN set, no rater keys present


In [21]:
t0 = time.perf_counter()
r = subprocess.run([PY, 'manage.py', 'infer', '--condition', ARM, '--config', str(CONFIG)],
                   check=False)
assert r.returncode == 0, f'{ARM} exited {r.returncode}'
GEN_SECONDS = round(time.perf_counter() - t0, 1)
print(f'{GEN_SECONDS / 60:.1f} min, finished {datetime.now(timezone.utc).isoformat()}')

sparse_knn: k=8 as up to 4 rarity + cosine for 1323 ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6091.93it/s]


  routes: {'full': 483, 'partial': 724, 'dense': 116}, mean rarity slots filled: 2.52


Loading weights: 100%|██████████| 339/339 [00:02<00:00, 154.96it/s]


Generating 1323 translations with Qwen/Qwen2.5-7B-Instruct (sparse_knn) ...
  5/1323
  10/1323
  15/1323
  20/1323
  25/1323
  30/1323
  35/1323
  40/1323
  45/1323
  50/1323
  55/1323
  60/1323
  65/1323
  70/1323
  75/1323
  80/1323
  85/1323
  90/1323
  95/1323
  100/1323
  105/1323
  110/1323
  115/1323
  120/1323
  125/1323
  130/1323
  135/1323
  140/1323
  145/1323
  150/1323
  155/1323
  160/1323
  165/1323
  170/1323
  175/1323
  180/1323
  185/1323
  190/1323
  195/1323
  200/1323
  205/1323
  210/1323
  215/1323
  220/1323
  225/1323
  230/1323
  235/1323
  240/1323
  245/1323
  250/1323
  255/1323
  260/1323
  265/1323
  270/1323
  275/1323
  280/1323
  285/1323
  290/1323
  295/1323
  300/1323
  305/1323
  310/1323
  315/1323
  320/1323
  325/1323
  330/1323
  335/1323
  340/1323
  345/1323
  350/1323
  355/1323
  360/1323
  365/1323
  370/1323
  375/1323
  380/1323
  385/1323
  390/1323
  395/1323
  400/1323
  405/1323
  410/1323
  415/1323
  420/1323
  425/1323
  430/132

---
## 7 — The output and its provenance

In [22]:
ARM_PATH = OUT / f'{ARM}_{SPLIT}.jsonl'
ARM_ROWS = [json.loads(x) for x in ARM_PATH.open(encoding='utf-8') if x.strip()]

assert len(ARM_ROWS) == len(VAL), f'{len(ARM_ROWS)} rows, expected {len(VAL)}'
assert [r['input'] for r in ARM_ROWS] == SRC, 'source order differs from the eval file'
assert all(r['condition'] == ARM for r in ARM_ROWS), 'mislabelled rows'
blank = [i for i, r in enumerate(ARM_ROWS) if not r['prediction'].strip()]
errored = [i for i, r in enumerate(ARM_ROWS) if 'error' in r]
assert not errored, f'{len(errored)} segments recorded an error: {errored[:5]}'
print(f'{len(ARM_ROWS)} rows, {len(blank)} blank predictions, {len(errored)} errors')

1323 rows, 0 blank predictions, 0 errors


In [23]:
USAGE = json.loads((OUT / f'{ARM}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
PROV = USAGE['provenance']

assert PROV['index_dir'] == RETR['index_dir'], PROV['index_dir']
assert PROV['rarity_sha256'] == RARITY_SHA, f"{PROV['rarity_sha256']} != {RARITY_SHA}"
assert PROV['k'] == RETR['k'] and PROV['m'] == SPA['m'], PROV
assert (PROV['df_min'], PROV['df_max']) == (RAR['df_min'], RAR['df_max']), PROV
assert PROV['min_query_terms'] == SPA['min_query_terms'], PROV
assert PROV['redundancy'] == SPA['redundancy'], PROV
assert PROV['ordering'] == PROMPT['ordering'], PROV
print(json.dumps(PROV, indent=2))
print(f"{USAGE['calls']} calls, ${USAGE.get('cost_usd', 0.0):.2f} — local weights, nothing paid")

{
  "k": 8,
  "ordering": "most_similar_last",
  "index_dir": "data/knn_index",
  "m": 4,
  "min_query_terms": 1,
  "redundancy": 0.3,
  "df_min": 2,
  "df_max": 20,
  "rarity_file": "results/rarity_train.json",
  "rarity_sha256": "47da96ea667ee0951278d16812b3fd49e4dc9f54aaf34bfbc4562765a1977ca5"
}
1323 calls, $0.00 — local weights, nothing paid


---
## 8 — Divergence from knn_fewshot

In [24]:
ARM_PRED = [r['prediction'] for r in ARM_ROWS]
BASE_PRED = [r['prediction'] for r in BASE_ROWS]
DIFFERS = np.array([a != b for a, b in zip(ARM_PRED, BASE_PRED)])
ROUTE = np.array([t['route'] for t in TRACES])
N_SPARSE = np.array([t['n_sparse'] for t in TRACES])

share = DIFFERS.mean()
print(f'{DIFFERS.sum()}/{len(DIFFERS)} predictions differ ({share:.1%}); '
      f'routed fraction is {routed / len(TRACES):.1%}')
for r in ('full', 'partial', 'dense'):
    sel = ROUTE == r
    print(f'  {r:8s} n={sel.sum():5d}  differ {DIFFERS[sel].mean():.1%}')

1154/1323 predictions differ (87.2%); routed fraction is 91.2%
  full     n=  483  differ 99.8%
  partial  n=  724  differ 88.7%
  dense    n=  116  differ 25.9%


In [25]:
dense_diff = DIFFERS[ROUTE == 'dense'].mean()
assert share >= 0.40, (
    f'only {share:.1%} of predictions differ against a {routed / len(TRACES):.1%} routed '
    f'fraction; the rarity channel is not reaching the prompt')
if not 0.60 <= share <= 0.98:
    print(f'NOTE: {share:.1%} differ, outside the 60-98% this design usually lands in. '
          f'Worth reading the per-route rows above before treating the deltas as selection.')
if dense_diff:
    print(f'WARNING: {dense_diff:.1%} of dense-routed segments differ despite an identical '
          f'prompt — greedy decoding did not reproduce across sessions, so read the deltas '
          f'below against this floor, not against zero.')
else:
    print('every dense-routed segment reproduces the baseline exactly; the deltas below '
          'carry no decode noise')

In [26]:
i = int(np.flatnonzero((ROUTE == 'full') & DIFFERS)[0])
print('SOURCE  :', SRC[i][:110])
print('terms   :', TRACES[i]['query_terms'][:8], f"coverage {TRACES[i]['coverage']}")
print(f'{BASELINE:12s}:', BASE_PRED[i][:200])
print(f'{ARM:12s}:', ARM_PRED[i][:200])

SOURCE  : جواهر الأسرار فی معارج الأسفار لمن اراد ان یتقرّب بالله المقتدر الغفّار فهنیاً للأبرار الّذین یشربون من هذه ال
terms   : ['الأسرار', 'الأسفار', 'الأنهار', 'الغفار', 'معارج', 'یتقرب', 'یشربون'] coverage 0.5912
knn_fewshot : Jewels of the mysteries in the ascent of journeys for him who desireth to draw nigh to God, the Almighty, the Forgiver; blessed indeed are the righteous who drink of these rivers.
sparse_knn  : Jewels of mysteries in the ascent of journeys for him who desireth to draw nigh unto the Almighty, the Pardoner of sins; verily, a bountiful provision for the righteous who drink of these rivers.


---
## 9 — Stage D: scoring

In [ ]:
CONDS = [BASELINE, ARM]
!{PY} manage.py eval --conditions {' '.join(CONDS)} --split {SPLIT}

In [ ]:
from src.eval.quick import score

SURFACE = {c: score(c, OUT, SPLIT) for c in CONDS}
print(f"{'condition':14s} {'chrF':>8s} {'BLEU':>8s} {'markers/seg':>12s}")
for cond in CONDS:
    s = SURFACE[cond]
    print(f"{cond:14s} {s['chrF']:8.2f} {s['BLEU']:8.2f} {s['marker_rate']:12.2f}")
print(f"gold targets carry {SURFACE[BASELINE]['ref_marker_rate']:.2f} markers per segment")

### COMET

In [ ]:
if not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install', '-q']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)

COMET_PATH = f'results/comet_{SPLIT}.json'
PRIOR_COMET = set(json.loads(Path(COMET_PATH).read_text(encoding='utf-8')))
assert BASELINE in PRIOR_COMET, f'{BASELINE} is not in {COMET_PATH} to pair against'

r = subprocess.run([COMET_PY, 'manage.py', 'comet', '--conditions', ARM, '--split', SPLIT,
                    '--results_path', COMET_PATH, '--batch_size', '16'], check=False)
assert r.returncode == 0, f'comet exited {r.returncode}'

In [ ]:
COMET = json.loads(Path(COMET_PATH).read_text(encoding='utf-8'))
assert PRIOR_COMET <= set(COMET), f'lost from {COMET_PATH}: {sorted(PRIOR_COMET - set(COMET))}'
ref, arm = COMET[BASELINE], COMET[ARM]
assert arm['n'] == len(VAL), arm['n']
assert arm['model'] == ref['model'], (arm['model'], ref['model'])
assert arm['sources'] == ref['sources'], 'the two conditions are not paired segment for segment'
print(f"{ref['model']}: {BASELINE} {ref['system']:.4f} -> {ARM} {arm['system']:.4f}")

### Register fit

In [ ]:
!{PY} manage.py stylometrics --conditions {' '.join(CONDS)} --split {SPLIT} --targets-split train

In [ ]:
STYLO_PATH = f'results/stylometrics_ci_{ARM}_{SPLIT}.json'
!{PY} manage.py stylometrics_ci --split {SPLIT} --conditions {' '.join(CONDS)} \
    --n_resamples {N_STYLO} --alpha {ALPHA} --seed {SEED} --results_path {STYLO_PATH}

In [ ]:
STYLO = json.loads(Path(STYLO_PATH).read_text(encoding='utf-8'))
old = json.loads(Path(f'results/stylometrics_ci_{SPLIT}.json').read_text(encoding='utf-8'))
a, b = STYLO['cells'][BASELINE], old['cells'][BASELINE]
assert a['stylo_dist'] == b['stylo_dist'] and a['z'] == b['z'], f'{BASELINE} moved between passes'
print(f"{BASELINE} reproduces the committed row: stylo_dist {a['stylo_dist']:.4f}")
for cond in CONDS:
    print(f"  {cond:14s} stylo_dist {STYLO['cells'][cond]['stylo_dist']:.4f}")

---
## 10 — Phi (paid)

`knn_fewshot` already carries Phi from the committed pass, so only the arm is bought:
one call per segment on the primary rater. The pilot measures the per-call cost before
the full pass is authorised.

In [ ]:
JUDGE_CFG = 'configs/judge_eval.yaml'
JUDGE_RESULTS = f'results/judge_{SPLIT}.json'
JUDGE_USAGE = f'results/judge_{SPLIT}_usage.json'
JUDGE_CI_PATH = f'results/judge_ci_{ARM}_{SPLIT}.json'

PRIOR_JUDGE = json.loads(Path(JUDGE_RESULTS).read_text(encoding='utf-8'))
PRIOR_USAGE = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
PRIOR_SPEND, PRIOR_CALLS = (PRIOR_USAGE['cumulative'][k] for k in ('cost_usd', 'calls'))

assert BASELINE in PRIOR_JUDGE, f'{BASELINE} has no Phi to compare against'
BUY = [c for c in CONDS if c not in PRIOR_JUDGE]
PER_CALL = PRIOR_USAGE['cumulative']['cost_usd'] / PRIOR_USAGE['cumulative']['calls']
N_CALLS = len(VAL) * len(BUY)
PROJECTED = PER_CALL * N_CALLS
print(f"buying Phi for {BUY or 'nothing'}: {N_CALLS} calls at ${PER_CALL:.5f} "
      f"= ${PROJECTED:.2f} projected (cumulative judge spend ${PRIOR_SPEND:.2f})")

In [ ]:
# Left False so a top-to-bottom re-run cannot authorise itself.
SPEND_OK = False
BUDGET_USD = 1.80
N_PILOT = 25

assert PROJECTED <= BUDGET_USD, f'projection ${PROJECTED:.2f} exceeds the ${BUDGET_USD:.2f} cap'
print(f'authorised {SPEND_OK}   cap ${BUDGET_USD:.2f}   pilot {N_PILOT} x {len(BUY)} '
      f'(${PER_CALL * N_PILOT * len(BUY):.3f})')

In [ ]:
if not BUY:
    print('nothing to buy: every condition already carries Phi')
else:
    assert SPEND_OK, 'set SPEND_OK = True in the cell above to authorise the pilot'
    r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', *BUY, '--split', SPLIT,
                        '--config', JUDGE_CFG, '--limit', str(N_PILOT)], check=False)
    assert r.returncode == 0, f'pilot exited {r.returncode}'

In [ ]:
_u = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
if _u['conditions'] != sorted(BUY) or _u['limit'] != N_PILOT:
    REVISED = PROJECTED
    print(f"{JUDGE_USAGE} holds {_u['conditions']} at limit {_u['limit']}, not this pilot: "
          f'${PROJECTED:.2f} stands')
else:
    pilot = _u['session']
    REVISED = pilot['cost_usd'] / pilot['calls'] * N_CALLS
    print(f"pilot {pilot['calls']} calls at ${pilot['cost_usd'] / pilot['calls']:.5f} "
          f'-> ${REVISED:.2f} for the full pass')
assert REVISED <= BUDGET_USD, f'revised ${REVISED:.2f} exceeds the ${BUDGET_USD:.2f} cap'

In [ ]:
# The client is built on first call, so re-running over a complete cache makes no request.
if not BUY:
    print('nothing to buy; the results file already carries every condition')
else:
    assert SPEND_OK, 'set SPEND_OK = True to authorise the full pass'
    r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', *BUY, '--split', SPLIT,
                        '--config', JUDGE_CFG], check=False)
    assert r.returncode == 0, f'judge exited {r.returncode}'

In [ ]:
JUDGE = json.loads(Path(JUDGE_RESULTS).read_text(encoding='utf-8'))
HAVE_PHI = ARM in JUDGE
if HAVE_PHI:
    r = subprocess.run([PY, 'manage.py', 'judge_ci', '--split', SPLIT, '--conditions', *CONDS,
                        '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                        '--results_path', JUDGE_CI_PATH], check=False)
    assert r.returncode == 0, f'judge_ci exited {r.returncode}'
else:
    print(f'{ARM} carries no Phi; sections 10 and 11 report adequacy and register only')

In [ ]:
lost = sorted(set(PRIOR_JUDGE) - set(JUDGE))
assert not lost, f'lost from {JUDGE_RESULTS}: {lost}'
if HAVE_PHI:
    assert JUDGE[ARM]['model'] == JUDGE[BASELINE]['model'], 'two raters, not one'

usage = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
assert usage['priced'], 'the judge model has no pricing table; cost_usd is a floor, not a bill'
SPENT = usage['cumulative']['cost_usd'] - PRIOR_SPEND
PAID_CALLS = usage['cumulative']['calls'] - PRIOR_CALLS
assert SPENT <= BUDGET_USD, f'${SPENT:.2f} spent against a ${BUDGET_USD:.2f} cap'

METRICS = ['chrf', 'bleu', 'comet'] + (['judge'] if HAVE_PHI else [])
print(f"{PAID_CALLS} paid calls, ${SPENT:.2f} on {usage['model']} this session; "
      f"cumulative ${usage['cumulative']['cost_usd']:.2f}")
print('reading out on', ', '.join(METRICS))

---
## 11 — The paired bootstrap

In [ ]:
BOOT_PATHS = {}
for metric in METRICS:
    path = f'results/bootstrap_{metric}_{ARM}_{SPLIT}.json'
    r = subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', metric,
                        '--conditions', *CONDS, '--split', SPLIT, '--pairs', f'{ARM}:{BASELINE}',
                        '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                        '--out', path], check=False)
    assert r.returncode == 0, f'{metric} bootstrap exited {r.returncode}'
    BOOT_PATHS[metric] = path

---
## 12 — Read-out by dose

The 116 dense-routed segments got the baseline's prompt, so they can only pull a delta
toward zero. Reading the arm three ways separates the treated population from the diluted one.
The detection floor is the smallest difference this design resolves at n=1,323; on a subset
it rises with 1/sqrt(n), which is why the full-dose column needs a larger effect to clear it.

In [ ]:
from src.eval.bootstrap import _load_segment_scores, paired_bootstrap

ROUTE_JSON = json.loads(ROUTING.read_text(encoding='utf-8'))['segments']
assert len(ROUTE_JSON) == len(VAL), len(ROUTE_JSON)

STRATA = {
    f'full dose (n_sparse = {SLOTS})': [i for i, t in enumerate(ROUTE_JSON)
                                        if t['n_sparse'] == SLOTS],
    'all routed': [i for i, t in enumerate(ROUTE_JSON) if t['route'] != 'dense'],
    'all segments': list(range(len(VAL))),
}
# Declared at n=1,323; scaled by 1/sqrt(n) for the subsets.
FLOOR = {'judge': 0.058, 'comet': 0.005}
print({k: len(v) for k, v in STRATA.items()})

In [ ]:
SCORES = {}
for metric in METRICS:
    scores, sources = _load_segment_scores(metric, CONDS, OUT, SPLIT, None)
    for cond in CONDS:
        assert cond in scores, f'{metric}: {cond} has no per-segment scores'
        assert len(scores[cond]) == len(VAL), (metric, cond, len(scores[cond]))
        if sources.get(cond) is not None:
            assert sources[cond] == SRC, f'{metric}/{cond}: segment order is not the eval order'
    SCORES[metric] = scores
print('per-segment scores aligned to the eval order for', ', '.join(SCORES))

In [ ]:
PLACES = {'chrf': 2, 'bleu': 2, 'comet': 4, 'judge': 4}

def delta(metric, idx):
    a = [SCORES[metric][ARM][i] for i in idx]
    b = [SCORES[metric][BASELINE][i] for i in idx]
    return paired_bootstrap(a, b, n_resamples=N_BOOT, alpha=ALPHA, seed=SEED)

for metric in METRICS:
    p = PLACES[metric]
    print(f'\n{metric}   {ARM} - {BASELINE}')
    for label, idx in STRATA.items():
        d = delta(metric, idx)
        mark = '*' if d['significant'] else ' '
        line = (f"  {label:26s} n={d['n']:5d}  {d['diff']:+.{p}f} "
                f"[{d['ci_low']:+.{p}f}, {d['ci_high']:+.{p}f}]  p={d['p_value']:.4f} {mark}")
        if metric in FLOOR:
            f = FLOOR[metric] * (len(VAL) / d['n']) ** 0.5
            line += f"  floor {f:.{p}f}{'' if abs(d['diff']) >= f else '  (under)'}"
        print(line)

In [ ]:
# The CLI's own table, for the pair the command line names.
for metric, path in BOOT_PATHS.items():
    rec = next(r for r in json.loads(Path(path).read_text(encoding='utf-8'))['comparisons']
               if (r['a'], r['b']) == (ARM, BASELINE))
    p = PLACES[metric]
    print(f"{metric:6s} {rec['diff']:+.{p}f} [{rec['ci_low']:+.{p}f}, {rec['ci_high']:+.{p}f}] "
          f"p={rec['p_value']:.4f} n={rec['n']}")

---
## 13 — Seal

In [ ]:
assert not list(OUT.glob('*_test.jsonl')), 'a test-split output exists'
assert not list(Path('results').glob('*_test.json')), 'a test-split result exists'
assert USAGE.get('cost_usd', 0.0) == 0.0, USAGE
print(f'generation: {USAGE["calls"]} calls, $0.00 (local weights)')
print(f'judge: {PAID_CALLS} paid calls, ${SPENT:.2f}')

In [ ]:
ARTEFACTS = ([str(ARM_PATH), str(OUT / f'{ARM}_{SPLIT}_usage.json'), str(ROUTING),
              COMET_PATH, STYLO_PATH, *BOOT_PATHS.values()]
             + ([JUDGE_RESULTS, JUDGE_CI_PATH] if HAVE_PHI else []))
for f in ARTEFACTS:
    assert Path(f).exists(), f
!git status --short {' '.join(ARTEFACTS)}

In [ ]:
import zipfile

BUNDLE = Path(f'{ARM}_{SPLIT}.zip')
with zipfile.ZipFile(BUNDLE, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ARTEFACTS:
        z.write(f, f)
print(f'{BUNDLE}  {BUNDLE.stat().st_size / 1e6:.1f} MB')